<a href="https://colab.research.google.com/github/saumyasrivastava22/Colab-Notebooks/blob/main/Hello_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# One-time setup

In [44]:
pip install streamlit openai streamlit-javascript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 518.4/518.4 kB 17.2 MB/s eta 0:00:00


# Policy Management App


*   Create a streamlit app which allows user to switch roles. Available role - product manager, customer support,  operations
*   A Product Manager should be able to upload a file temporarily to test their policy text only. This should not interfere with the day to day job of customer support and operations.
* Customer support should be able to upload policy docs, ask questions and publish the policy for others to use. The response should allow copy answer to clipboard option.
* Operations should be able to ask questions and be able to copy answer to clipboard.



In [81]:
%%writefile hello-agent-app.py
import os
import pandas as pd
import streamlit as st
import streamlit.components.v1 as components
from google.colab import userdata
from io import StringIO
from openai import OpenAI
import json
import time

# --- Constants ---
SUPPORTED_FILE_TYPES = ["csv", "xlsx"]
COMMON_POLICY_SYSTEM_PROMPT = "You are a helpful assistant. Answer questions strictly based on the provided policy documents. Policy documents are provided below:\n"

# Updated pricing per 1M tokens
MODEL_PRICING = {
    "gpt-4o-mini": {"input": 0.15, "output": 0.60},
    "gpt-4o": {"input": 2.50, "output": 10.00},
    "gpt-5": {"input": 5.00, "output": 15.00},
    "omni-moderation-latest": {"input": 0.10, "output": 0.10}
}

# --- Helper Functions ---
def get_openai_client():
  client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
  return client

def upload_file(file):
  if file is not None:
    file_extension = file.name.split('.')[-1].lower()
    if file_extension == 'csv':
        dataframe = pd.read_csv(file)
    elif file_extension == 'xlsx':
        dataframe = pd.read_excel(file)
    else:
        st.error(f"Unsupported file type: .{file_extension}")
        return None
    return dataframe
  return None

def upload_files(*files):
  all_dataframes = []
  for file_obj in files:
    df = upload_file(file_obj)
    if df is not None:
      all_dataframes.append((file_obj.name, df))
  return all_dataframes

@st.cache_data(show_spinner=False, ttl=600)
def get_llm_response(model, messages):
    if 'llm_stats' not in st.session_state:
        st.session_state.llm_stats = {'total_calls': 0, 'models': {}, 'est_cost': 0.0}

    st.session_state.llm_stats['total_calls'] += 1
    st.session_state.llm_stats['models'][model] = st.session_state.llm_stats['models'].get(model, 0) + 1

    response = client.chat.completions.create(
        model=model,
        messages=messages
    )

    # Rough estimation: input tokens ~1500, output ~500 for a typical policy query
    prices = MODEL_PRICING.get(model, MODEL_PRICING['gpt-4o-mini'])
    cost = (prices['input'] * 1500 / 1e6) + (prices['output'] * 500 / 1e6)
    st.session_state.llm_stats['est_cost'] += cost

    return response.choices[0].message.content

def _display_answer_and_copy_button(session_state_answer_key, button_key_prefix):
    if st.session_state.get(session_state_answer_key):
        st.success("Answer received:")
        answer_text = st.session_state[session_state_answer_key]
        st.write(answer_text)

        if st.button("Copy to clipboard", key=f"btn_{button_key_prefix}"):
            safe_text = json.dumps(answer_text)
            copy_js = f"""
                <script>
                function doCopy() {{
                    const text = {safe_text};
                    var textArea = document.createElement("textarea");
                    textArea.value = text;
                    textArea.style.position = "fixed";
                    textArea.style.left = "-9999px";
                    textArea.style.top = "0";
                    if (document.body) {{
                        document.body.appendChild(textArea);
                        textArea.focus();
                        textArea.select();
                        try {{ document.execCommand('copy'); alert("Copied to clipboard!"); }} catch (err) {{ }}
                        document.body.removeChild(textArea);
                    }}
                }}
                doCopy();
                </script>
            """
            components.html(copy_js, height=0, width=0)

# --- App Initialization ---
client = get_openai_client()
st.set_page_config(page_title="Policy Management App", layout="wide")
st.title("Policy Management and Q&A")

if 'llm_stats' not in st.session_state: st.session_state.llm_stats = {'total_calls': 0, 'models': {}, 'est_cost': 0.0}
if 'published_policies_context' not in st.session_state: st.session_state.published_policies_context = ""
if 'cs_ans' not in st.session_state: st.session_state.cs_ans = ""
if 'ops_ans' not in st.session_state: st.session_state.ops_ans = ""

# Sidebar Metrics & Config
st.sidebar.header("Configuration")
active_model = st.sidebar.selectbox("Select Active Model:", list(MODEL_PRICING.keys()))

st.sidebar.header("Detailed Usage Breakdown")
st.sidebar.metric("Total LLM Calls", st.session_state.llm_stats['total_calls'])
st.sidebar.write("**Calls by Model:**")
for mod, count in st.session_state.llm_stats['models'].items():
    st.sidebar.write(f"- {mod}: {count}")
st.sidebar.write(f"**Estimated Total Cost:** ${st.session_state.llm_stats['est_cost']:.5f}")

# Common Cache Counter (Counting non-empty responses in common shared roles)
common_cache_count = sum(1 for k in ['cs_ans', 'ops_ans'] if st.session_state[k])
st.sidebar.divider()
st.sidebar.write(f"**Common Cache Entries:** {common_cache_count}")

roles = { "Product Manager": "pm", "Customer Support": "cs", "Operations": "ops" }
selected_role = roles[st.sidebar.selectbox("Select Role:", list(roles.keys()))]

if selected_role == "pm":
    f = st.file_uploader("Upload test policy", type=SUPPORTED_FILE_TYPES)
    if f:
        df = upload_file(f)
        st.session_state.pm_context = df.to_csv(index=False)
        st.dataframe(df.head())
    q = st.text_area("Test Question")
    if st.button("Run Test"):
        ctx = st.session_state.get('pm_context', st.session_state.published_policies_context)
        st.session_state.pm_ans = get_llm_response(active_model, [{"role": "system", "content": COMMON_POLICY_SYSTEM_PROMPT + ctx}, {"role": "user", "content": q}])
    _display_answer_and_copy_button('pm_ans', 'pm')

elif selected_role == "cs":
    files = st.file_uploader("Upload policies", accept_multiple_files=True, type=SUPPORTED_FILE_TYPES)
    if files:
        named_dfs = upload_files(*files)
        tabs = st.tabs([n for n, d in named_dfs])
        for i, (n, d) in enumerate(named_dfs):
            with tabs[i]: st.dataframe(d.head())
        if st.button("Publish"):
            st.session_state.published_policies_context = pd.concat([d for n, d in named_dfs]).to_csv(index=False)
            st.success("Published!")
    q = st.text_area("Ask CS")
    if st.button("Get Answer"):
        st.session_state.cs_ans = get_llm_response(active_model, [{"role": "system", "content": COMMON_POLICY_SYSTEM_PROMPT + st.session_state.published_policies_context}, {"role": "user", "content": q}])
    _display_answer_and_copy_button('cs_ans', 'cs')

elif selected_role == "ops":
    q = st.text_area("Ask Ops")
    if st.button("Get Answer"):
        st.session_state.ops_ans = get_llm_response(active_model, [{"role": "system", "content": COMMON_POLICY_SYSTEM_PROMPT + st.session_state.published_policies_context}, {"role": "user", "content": q}])
    _display_answer_and_copy_button('ops_ans', 'ops')


Overwriting hello-agent-app.py


The Streamlit app might not be running stably or the port might be occupied. Let's ensure the port is clear and then run Streamlit more persistently.

Now that Streamlit is running persistently in the background, let's start `localtunnel` to get a public URL. This time, `localtunnel` will run in the foreground and display the URL clearly.

# Hosting the App

In [ ]:
# Install ngrok
!pip install pyngrok

The previous `ngrok` attempt still led to an unreachable site. Let's make the Streamlit startup more robust and ensure `ngrok` connects only after Streamlit is ready. We'll also confirm the `NGROK_AUTH_TOKEN` is correctly loaded.

In [72]:
import time
import subprocess
from pyngrok import ngrok
import os
from google.colab import userdata

# 1. Kill any existing Streamlit or ngrok processes
print("Killing any existing Streamlit and ngrok processes...")
!pkill -f streamlit || true
ngrok.kill()

# 2. Start Streamlit in the background with nohup and log output
print("Starting Streamlit app in background...")
# Make sure to use the correct filename `hello-agent-app.py`
subprocess.Popen([
    "nohup", "streamlit", "run", "hello-agent-app.py",
    "--server.port", "8501",
    "--server.enableCORS", "false",
    "--server.enableXsrfProtection", "false"
], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, preexec_fn=os.setpgrp)

# 3. Verify NGROK_AUTH_TOKEN and authenticate
NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')
if NGROK_AUTH_TOKEN is None:
    print("NGROK_AUTH_TOKEN not found in Colab secrets. Please set it.")
    # You may want to stop execution or prompt the user here
else:
    print("Authenticating ngrok...")
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# 4. Give Streamlit some time to start
print("Waiting 10 seconds for Streamlit to fully start...")
time.sleep(10)

# 5. Open a ngrok tunnel to the Streamlit port
try:
    print("Attempting to establish ngrok tunnel...")
    public_url = ngrok.connect(8501)
    print(f"\nYour Streamlit app is available at: {public_url}")
    print("Please click the link above.")
except Exception as e:
    print(f"Error creating ngrok tunnel: {e}")
    print("Please ensure Streamlit is running and your NGROK_AUTH_TOKEN is correct.")


Killing any existing Streamlit and ngrok processes...
^C
Starting Streamlit app in background...
Authenticating ngrok...
Waiting 10 seconds for Streamlit to fully start...
Attempting to establish ngrok tunnel...

Your Streamlit app is available at: NgrokTunnel: "https://spore-supervise-stays.ngrok-free.dev" -> "http://localhost:8501"
Please click the link above.


The previous `ngrok` attempt still led to an unreachable site. Let's make the Streamlit startup more robust and ensure `ngrok` connects only after Streamlit is ready. We'll also confirm the `NGROK_AUTH_TOKEN` is correctly loaded.

# Appendix

In [ ]:
import os
from openai import OpenAI
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
client.responses.connect()
client.models.list()

SyncPage[Model](data=[Model(id='text-embedding-ada-002', created=1671217299, object='model', owned_by='openai-internal'), Model(id='whisper-1', created=1677532384, object='model', owned_by='openai-internal'), Model(id='tts-1', created=1681940951, object='model', owned_by='openai-internal'), Model(id='text-embedding-3-small', created=1705948997, object='model', owned_by='system'), Model(id='text-embedding-3-large', created=1705953180, object='model', owned_by='system'), Model(id='gpt-4o', created=1715367049, object='model', owned_by='system'), Model(id='gpt-4o-mini', created=1721172741, object='model', owned_by='system'), Model(id='omni-moderation-latest', created=1731689265, object='model', owned_by='system'), Model(id='gpt-5', created=1754425777, object='model', owned_by='system')], object='list')